> 在线讲义：[第 21 章 · 2026 开源 VLA：读一份 config，选一个能微调的模型](https://wikieden.github.io/vla-from-scratch/lessons/21-open-models-2026.html) · 本书《VLA 从零到 π0》

# 第 21 章 · 2026 开源 VLA：读一份 config，选一个能微调的模型

配套讲义：仓库里 `vla-deep-dive/notes/lessons/21-open-models-2026.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `MODELS` | 11 个模型的五个字段 |
| 第 2 步 | `LINGBOT_CANONICAL`、`canonical_dim(...)` | 55 |
| 第 2 步 | `quantile_normalize(x, q01, q99)` | 把 [q01, q99] 映到 [−1, 1] |
| 第 3 步 | `moe_active_fraction(32, 4)`、`moe_active_width(4, 512, 704)` | 12.5%；2752 |
| 第 4 步 | `finetune_gb(params_b, mode)` | 0.45B：7.2 / 0.9 GB；4B：64 / 8 GB |

数字取自 LeRobot policies/<名字>/configuration_*.py 的默认值；None 表示 config 里没有直接写。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MODELS = {
    "SmolVLA": ("SmolVLM2-500M（前 16 层）", 50, 32, 10),
    "π0.5": ("PaliGemma", 50, 32, 10),
    "X-VLA": ("Florence-2", 32, 20, 10),
    "Wall-OSS-0.5": ("Qwen2.5-VL", 32, 26, 10),
    "LingBot-VLA 2.0": ("Qwen3-VL-4B", 50, 55, 10),
    "EO-1": ("Qwen2.5-VL-3B", 16, 32, None),
    "EVO1": ("InternVL3-1B（14 层）", 50, 24, None),
    "DM0.5": ("OpenDM", 50, 32, None),
    "MolmoAct2": ("AI2 VLM", 30, None, 8),
    "VLA-JEPA": ("Qwen3-VL", 7, 7, None),
    "FastWAM": ("Wan2.2-TI2V-5B", 32, 7, 10),
}
LINGBOT_CANONICAL = {"arm.position": 14, "end.position": 14, "effector.position": 2, "hand.position": 12,
                     "waist.position": 4, "head.position": 2, "base.velocity": 3, "reserved.slots": 4}
BYTES_PER_PARAM = {"全参数": 16, "LoRA": 2, "只推理": 2}

**函数 `canonical_dim`**：统一空间的总格数。

In [ ]:
def canonical_dim(layout):
    """统一空间的总格数。"""
    return sum(layout.values())

**函数 `quantile_normalize`**：把 [q01, q99] 线性映到 [−1, 1]，超出的截到 ±1。

In [ ]:
def quantile_normalize(x, q01, q99):
    """把 [q01, q99] 线性映到 [−1, 1]，超出的截到 ±1。"""
    y = 2 * (x - q01) / (q99 - q01) - 1
    return max(-1.0, min(1.0, y))

**函数 `moe_active_fraction`**：每个 token 用到的路由专家比例。

In [ ]:
def moe_active_fraction(num_experts, top_k):
    """每个 token 用到的路由专家比例。"""
    return top_k / num_experts

**函数 `moe_active_width`**：每个 token 实际走过的 FFN 中间维：k 个专家 + 共享专家。

In [ ]:
def moe_active_width(top_k, expert_width, shared_width):
    """每个 token 实际走过的 FFN 中间维：k 个专家 + 共享专家。"""
    return top_k * expert_width + shared_width

**函数 `finetune_gb`**：只算权重和优化器状态的显存（GB）；params_b 以十亿参数计。

In [ ]:
def finetune_gb(params_b, mode):
    """只算权重和优化器状态的显存（GB）；params_b 以十亿参数计。"""
    return params_b * BYTES_PER_PARAM[mode]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 config 里的五个字段")
for name, (vlm, chunk, dim, steps) in MODELS.items():
    print(f"  {name:<16} 主干 {vlm:<24} chunk {chunk:>3}  动作维 {str(dim or '—'):>3}  去噪 {steps or '—'}")
flow10 = sum(1 for *_, s in MODELS.values() if s == 10)
print(f"  写明 10 步去噪的：{flow10} / {len(MODELS)}")

In [ ]:
print("\n第二步 LingBot-VLA 2.0 的统一空间与 q01/q99 归一化")
print(f"  {' + '.join(str(v) for v in LINGBOT_CANONICAL.values())} = {canonical_dim(LINGBOT_CANONICAL)}")
print(f"  q01 = −0.2、q99 = 0.6：x = 0.2 → {quantile_normalize(0.2, -0.2, 0.6):+.2f}，"
      f"x = 0.9 → {quantile_normalize(0.9, -0.2, 0.6):+.2f}（截断）")

In [ ]:
print("\n第三步 稀疏 MoE 动作专家")
print(f"  32 选 4：每个 token 用 {moe_active_fraction(32, 4):.1%} 的路由专家")
print(f"  4 × 512 + 704 = {moe_active_width(4, 512, 704)}（等于不用 MoE 时的 2752）")

In [ ]:
print("\n第四步 微调显存（只算权重和优化器）")
for p in (0.45, 3, 4):
    print(f"  {p}B：全参数 {finetune_gb(p, '全参数'):.1f} GB，LoRA {finetune_gb(p, 'LoRA'):.1f} GB")

## 3. 代码里才看得到的

1. `MODELS` 里的数都是 config 的**默认值**；同一个模型在 LIBERO 上微调时常改（比如 DM0.5 的 LIBERO 配方把 chunk 改成 10，README L30–L31）。
2. `finetune_gb` 只算权重和优化器状态，不算激活和 LoRA 的那一点可训参数；真实显存还要加激活，图像 token 多时激活可能比权重还大。
3. `quantile_normalize` 会把超出 [q01, q99] 的值截到 ±1；分位数之外的极端动作在训练时就被压扁了，所以动作分布很长尾的任务要小心。

## 4. 自己改一改

- 给 `MODELS` 加一列"总参数量"，从 README 里查着填。
- 用 `finetune_gb` 画出 0.1–8B 的全参数和 LoRA 曲线，标出 24 GB 和 80 GB 两条线。
- 改 `moe_active_width` 的参数，找出"计算量不变、专家数翻倍"的组合。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 看车先看行驶证：大脑、动作头、几步、几格、推几步；
> 收纳盒格子越细，越能装手和腰，接入就写清放哪格；
> 专科门诊挂四个号，医院大，看一个人用不了多少医生；
> 全参十六字节，LoRA 两字节，先量门框再搬家。